# 阅读与修改 PyTorch 项目代码

## 学习目标

从训练入口追踪数据、模型、loss、优化器和 checkpoint，使用签名与源码定位职责，并完成一个范围明确的模型修改。

## 概念模型

阅读项目时先找执行入口，再沿调用链确认输入输出契约。不要从目录中的每个文件顺序阅读；围绕一次训练的数据流建立地图。

In [ ]:
import inspect
from pathlib import Path
import torch
from torch import nn
from common import engine, checkpoint, models, runtime

root = Path.cwd()
print('course root:', root)
print('train signature:', inspect.signature(engine.train_one_epoch))
print('evaluate signature:', inspect.signature(engine.evaluate))
print('checkpoint signature:', inspect.signature(checkpoint.save_checkpoint))

### 实验 1：建立训练数据流地图

**实验目的**：用源码检查确认训练引擎包含设备搬运、清梯度、反向和更新，并理解 `EpochResult` 返回契约。

字符串断言适合作为阅读辅助，不是稳健的生产测试；真正修改共享引擎时应增加行为测试，覆盖训练/验证、空 loader 和不等长 batch。


In [ ]:
source = inspect.getsource(engine._run_epoch)
required_steps = ['inputs.to(device)', 'optimizer.zero_grad', 'loss.backward', 'optimizer.step']
for step in required_steps:
    assert step in source, step
print('engine contract:', required_steps)
print('returns:', inspect.signature(engine.EpochResult))

### 实验 2：理解模型边界

**实验目的**：从公开构造参数、features、classifier 和前向 shape 识别模块所有权与扩展点。修改前先确认分类头输入维度、通道数和 pooling 契约。

阅读顺序应从入口和公共 API 向内追踪，而不是先陷入实现细节。


In [ ]:
base = models.ImageClassifier(channels=1, num_classes=10)
sample = torch.randn(2, 1, 28, 28)
assert base(sample).shape == (2, 10)
print('feature modules:', list(base.features.named_children()))
print('classifier:', base.classifier)

### 实验 3：做一个局部修改并保护契约

**实验目的**：只替换分类层并启用 dropout，将输出类别改为 5，同时断言输出 shape 与模块存在。局部修改应保持特征提取器边界不变。

真实提交还应测试训练梯度、state_dict、checkpoint 兼容性和错误输入；避免顺手重构无关模块。


In [ ]:
modified = models.ImageClassifier(channels=1, num_classes=10, dropout=0.3)
modified.classifier[-1] = nn.Linear(32 * 4 * 4, 5)
output = modified(sample)
assert output.shape == (2, 5)
assert any(isinstance(module, nn.Dropout) for module in modified.modules())
print('modified logits:', output.shape, 'parameters:', sum(p.numel() for p in modified.parameters()))

## 官方教程补充

**对应官方源文件：** `beginner_source/basics/quickstart_tutorial.py`、`intermediate_source/torchvision_tutorial.py`、`intermediate_source/parametrizations.py`

阅读官方示例时先找外部契约：输入/标签、`nn.Module.forward`、loss、optimizer 和 artifact；再沿实际执行顺序追踪，而不是按文件顺序扫读。修改模型优先通过子模块、参数化或明确扩展点完成，并用 shape、梯度和 round-trip 测试保护。注册参数、buffer 与普通属性的差异会影响设备移动和 state dict。

**验证练习：** 找到上面源文件中的对应 API，先写出输入、输出和状态变化，再运行本 notebook 的相关实验；如果行为不同，优先检查本地 PyTorch 版本、设备能力和输入契约。

<!-- official-pytorch-supplement-v1 -->

## 检查点

指出训练入口、数据入口、模型边界、loss 和 checkpoint 分别位于哪里；解释为什么修改分类头不需要修改通用训练循环。

## 试一试

阅读 `examples/train_image_classifier.py`，画出它与 `common` 模块的调用关系；再增加一个模型参数并确认 CLI、checkpoint 和 shape 测试是否需要变化。

## 常见错误与调试

从所有文件第一行开始顺序阅读、只看类名不检查输入输出、修改共享 engine 解决模型内部问题、没有测试修改后的 shape、忽略 checkpoint 与旧模型结构的兼容性。